<a href="https://colab.research.google.com/github/drmedina9606/03MIAR-AlgoritmosOptimizacion/blob/main/Danny_Medina_Moncayo_Trabajo_Pr%C3%A1ctico_AO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algoritmos de optimización - Trabajo Práctico<br>
Nombre y Apellidos: Danny Medina Moncayo  <br>
Url: https://github.com/drmedina9606/03MIAR-AlgoritmosOptimizacion/Trabajo_Practico <br>
Google Colab: https://colab.research.google.com/drive/1uETCY60YE1XkKOO73pSHaICwc6pLb1Mz?usp=sharing <br>
Problema:
>- Configuración de Tribunales

Descripción del problema:

Se precisa configurar tribunales de evaluación para un grupo de 15 alumnos que desean presentar su Trabajo de Fin de Master (TFM).

Cada tribunal está compuesto por tres profesores, cada uno desempeñando uno de los siguientes roles: Presidente, Secretario, Vocal.

Los profesores han indicado su disponibilidad horaria para participar en los tribunales de 15h a 21h durante la semana del 15 al 19 de abril:
* **Número de profesores**: 10
* **Número de tribunales**: 15
* **Disponibilidad y roles**: https://docs.google.com/spreadsheets/d/1nGeFXiDXH7Cy-ed8oUdrDfAQxQOzJ_MmJ5fqOSY9o48/edit?usp=sharing
  - 1 indica que el profesor tiene disponibilidad
  - 0 en caso contrario

* Hay 15 alumnos, por lo que se deben configurar 15 tribunales buscando la configuración más equilibrada posible en cuanto a la cantidad de tribunales asignados a cada profesor, es decir, evitando que un profesor tenga muchos tribunales o otros pocos.
* Obviamente ningún profesor puede asistir a dos tribunales a la misma fecha/hora y no puede ser convocado a un tribunal al que no tiene disponibilidad.






                                        

In [1]:
import numpy as np
import itertools
import pandas as pd

In [2]:
# Matriz de disponibilidad transpuesta
disponibilidad = '''
0,1,0,1,1,1,0,1,1,1;
1,1,0,0,1,1,1,1,0,1;
1,1,1,1,0,1,1,1,1,0;
1,1,1,0,1,1,1,1,1,1;
0,0,0,1,0,1,1,1,0,1;
1,0,1,1,1,0,1,0,1,0;
1,0,1,0,1,0,1,0,0,1;
1,0,1,1,1,1,1,1,0,1;
0,1,1,0,1,1,1,1,1,0;
1,1,1,0,1,1,0,0,1,0;
1,1,0,1,0,1,1,1,1,0;
1,1,0,1,1,1,1,1,1,0;
1,0,1,1,1,1,0,1,1,0;
1,0,1,1,1,1,1,0,1,1;
1,1,1,0,1,1,0,1,1,1;
1,0,1,0,0,1,0,1,1,1;
0,0,1,1,1,0,1,1,0,0;
0,1,1,1,1,0,1,0,0,0;
1,1,1,1,0,1,0,0,0,1;
0,1,1,1,1,1,0,1,1,1;
1,0,1,1,1,1,1,1,1,1;
0,1,1,1,0,1,1,0,1,1;
1,1,0,0,1,1,0,1,1,0;
1,1,1,1,1,1,0,1,1,0;
1,1,1,1,1,0,0,1,1,1;
1,1,1,0,0,0,0,1,1,1;
1,1,0,1,1,1,1,1,1,1;
1,1,1,1,1,1,1,1,0,1;
1,1,0,1,1,1,1,0,1,1;
1,1,1,1,0,1,1,1,0,1;
1,1,1,1,1,1,1,1,1,1;
1,1,0,1,1,0,1,1,1,0;
1,1,1,1,1,1,1,0,1,0;
0,1,0,1,1,0,0,1,1,0;
0,1,1,0,0,1,1,0,1,1
'''
# Roles: P, S, V
roles = '''
1,1,1;
1,1,1;
1,0,1;
0,1,1;
1,1,1;
1,1,1;
0,1,1;
0,1,1;
1,1,1;
1,1,1
'''
matriz_disponibilidad = np.matrix(disponibilidad.replace(",", " "))
matriz_roles = np.matrix(roles.replace(",", " "))

#Modelo
- ¿Como represento el espacio de soluciones?
- ¿Cual es la función objetivo?
- ¿Como implemento las restricciones?

*¿Como represento el espacio de soluciones?*

El espacio de soluciones se va a representar con un diccionario $T$ de tribunales. Cada tribunal $T_i$ contiene la siguiente información:
$$ T_i = \{horario: \ i, \ profesores: (0,...1,...1,...1,...0)\}$$

*¿Cual es la función objetivo?*

La función objetivo es la suma de la cantidad de veces que aparece cada profesor en tribunales diferentes. Sea $n_j$ el conteo de tribunales distintos en los que aparece el profesor j, entonces se busca minimizar:
$$min \sum_{j=1}^{10} n_j$$

*¿Como implemento las restricciones?*

Se deben verificar 2 restricciones: disponibilidad y roles.

Se va a escoger presidente, secretario y vocal siempre en ese orden. Sea $T_i(p_j, p_k, p_l) = (0,0,...,1,1,1,...,0)$ la conformación del tribunal con los profesores $p_j, p_k, p_l$ en el horario $i$.
    
* *Restricción de disponibilidad:* Sea $D_i$ la fila $i$ de la matriz de disponibilidad. Entonces, el tribunal $T_i(p_j, p_k, p_l)$ satisface la disponibilidad de todos los profesores si:

  $$T_i(p_j, p_k, p_l) * D_i = 3$$
  $$ j, k, l \in \{1,2, ..., 10\}$$
  $$ i \in \{1,2,..., 35\}$$

* *Restricción de roles:* Sea $R_T = (r_{i,j})$ la sub-matriz correspondiente a los roles de los profesores $p_j, p_k, p_l$. Entonces, el tribunal es válido si:
$$(1)\sum_{i=1}^{3} = r_{i,1} > 0$$
$$(2)\sum_{i=1}^{3} = r_{i,2} > 1$$
Estas restricciones aseguran que los roles de presidente y secretario estén cubiertos porque el rol de vocal siempre está cubierto.

#Análisis
- ¿Que complejidad tiene el problema?. Orden de complejidad y Contabilizar el espacio de soluciones

*Contabilización del espacio de soluciones*

Sea $n$ la cantidad de tribunales a conformar y $m$ los horarios (hora-día) disponibles. Notar que se puede llevar a cabo más de untribunal en el mismo horario porque se cuenta con una cantidad tres veces mayor de profesores en comparación con el número mínimo de profesores necesarios para formar un tribunal. Entonces el espacio de soluciones se puede contabilizar por medio de combinaciones con repetición:
$$n=15$$
$$m=35$$
$$\# soluciones = CR_{15}^{35} = \frac{(35 + 15 - 1)!}{15!(35-1)!} > 1.5  \ billones$$

*Orden de complejidad*

Considerando que el espacio de soluciones está compuesto de las combinaciones con repetición de los posibles horarios, entonces se puede concluir que el órden de complejidad de este problema es factorial.
$$O(n) = n!$$

#Diseño
- ¿Que técnica utilizo?

Se utilizará un algoritmo voraz para resolver este problema.

- ¿Por qué?

Existe un conjunto de candidatos como solución que se pueden obtener a partir de la matriz de disponibilidad.

Se puede desarrollar una función para determinar el mejor candidato, en la implementación del algoritmo, se utiliza el conteo de profesores en los tribunales seleccionados hasta una determinada iteración.

Se puede desarrolar una función que comprueba si un subconjunto de candidatos es prometedor.

In [3]:
def es_valido(estado, matriz_roles):
    roles = np.array(matriz_roles[estado.astype(bool)][0].sum(axis=0))[0]
    if (roles[2] > 0) and (roles[1] > 1) and (estado.sum() == 3):
        return True
    else: return False

def verificar_horarios(matriz_disponibilidad, horario):
    disp_horario = np.array(matriz_disponibilidad[horario])[0]
    if disp_horario.sum() > 2:
        return horario
    else:
        horario += 1
        return horario

def generar_tribunal(matriz_disponibilidad, horario, matriz_roles, excluir_profesor):
    disponibilidad = np.array(matriz_disponibilidad[horario])[0]
    prof = np.array(range(1,11)) * disponibilidad
    prof_disp = prof[prof > 0] - 1
    prof_disp = prof_disp.tolist()
    combinaciones = list(itertools.combinations(prof_disp, 3))
    if len(excluir_profesor) > 0:
        mitad = len(combinaciones) // 2
        combinaciones = combinaciones[mitad: len(combinaciones)]
    for c in combinaciones:
        estado = np.zeros(10)
        estado[list(c)] = 1
        if es_valido(estado, matriz_roles) == True:
            break
        return estado

def construir_tribunales(n_tribunales = 15,
                         matriz_disponibilidad = matriz_disponibilidad,
                         matriz_roles = matriz_roles):
    i = 0
    horario = 0
    matriz_estados = np.zeros((15,10))
    soluciones = {}
    excluir_profesor = []
    while len(soluciones.keys()) < n_tribunales:
        repetidos = matriz_estados.sum(axis=0)
        if repetidos.max() > 1:
            excluir_profesor = np.argwhere(repetidos == repetidos.max()).flatten().tolist()
            excluir_profesor = excluir_profesor[0:2]
        horario = verificar_horarios(matriz_disponibilidad=matriz_disponibilidad,
                                     horario=horario)
        estado = generar_tribunal(matriz_disponibilidad=matriz_disponibilidad,
                                  horario=horario,
                                  matriz_roles=matriz_roles,
                                  excluir_profesor=excluir_profesor)
        soluciones['Tribunal_' + str(i)] = horario
        matriz_estados[i] = estado
        matriz_disponibilidad[horario, estado.astype(bool)] = 0
        i += 1
    return soluciones, matriz_estados

soluciones, matriz_estados = construir_tribunales(15, matriz_disponibilidad, matriz_roles)

In [4]:
print(matriz_estados)
print(matriz_estados.sum(axis=0))
print(matriz_estados.sum(axis=1))
repetidos = matriz_estados.sum(axis=0)

[[0. 1. 0. 1. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 1. 1. 0.]
 [1. 1. 0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 1. 0. 1.]
 [0. 1. 0. 1. 0. 0. 0. 1. 0. 0.]
 [1. 0. 0. 0. 0. 0. 1. 0. 1. 0.]
 [0. 1. 0. 0. 0. 1. 0. 0. 0. 1.]
 [0. 0. 1. 0. 1. 0. 1. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0. 1. 1. 0.]
 [0. 0. 0. 1. 0. 0. 0. 1. 0. 1.]
 [0. 0. 1. 1. 1. 0. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 1. 0. 1. 0.]
 [1. 0. 0. 0. 0. 0. 1. 0. 0. 1.]
 [0. 0. 1. 0. 1. 0. 0. 1. 0. 0.]
 [1. 0. 0. 0. 0. 0. 1. 0. 0. 1.]]
[6. 4. 3. 4. 5. 3. 5. 6. 4. 5.]
[3. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3.]


In [5]:
soluciones

{'Tribunal_0': 0,
 'Tribunal_1': 0,
 'Tribunal_2': 1,
 'Tribunal_3': 1,
 'Tribunal_4': 2,
 'Tribunal_5': 2,
 'Tribunal_6': 3,
 'Tribunal_7': 3,
 'Tribunal_8': 3,
 'Tribunal_9': 4,
 'Tribunal_10': 5,
 'Tribunal_11': 5,
 'Tribunal_12': 6,
 'Tribunal_13': 7,
 'Tribunal_14': 7}

In [6]:
horario_dict = {}
i = 0
for fecha in range(15, 20):
    for hora in range(15, 22):
        horario_dict[i] = [fecha, hora]
        i += 1

In [9]:
profesores_dict = np.array(['RRD', 'QYV', 'LHL', 'HLC', 'MSB', 'PMQ', 'QWF', 'EBB', 'IOE', 'IOA'])

In [10]:
config_tribunales = []
for i in range(15):
    config_tribunales.append(list(profesores_dict[matriz_estados[i,:].astype(bool)]))
config_tribunales

[[np.str_('QYV'), np.str_('HLC'), np.str_('MSB')],
 [np.str_('PMQ'), np.str_('EBB'), np.str_('IOE')],
 [np.str_('RRD'), np.str_('QYV'), np.str_('MSB')],
 [np.str_('PMQ'), np.str_('EBB'), np.str_('IOA')],
 [np.str_('QYV'), np.str_('HLC'), np.str_('EBB')],
 [np.str_('RRD'), np.str_('QWF'), np.str_('IOE')],
 [np.str_('QYV'), np.str_('PMQ'), np.str_('IOA')],
 [np.str_('LHL'), np.str_('MSB'), np.str_('QWF')],
 [np.str_('RRD'), np.str_('EBB'), np.str_('IOE')],
 [np.str_('HLC'), np.str_('EBB'), np.str_('IOA')],
 [np.str_('LHL'), np.str_('HLC'), np.str_('MSB')],
 [np.str_('RRD'), np.str_('QWF'), np.str_('IOE')],
 [np.str_('RRD'), np.str_('QWF'), np.str_('IOA')],
 [np.str_('LHL'), np.str_('MSB'), np.str_('EBB')],
 [np.str_('RRD'), np.str_('QWF'), np.str_('IOA')]]

In [11]:
df_resultados_horario = pd.DataFrame.from_dict(soluciones, orient='index')
df_resultados_horario = pd.DataFrame(df_resultados_horario[0].map(horario_dict))
df_resultados_horario.columns = ['Fecha_Hora']
df_resultados_horario = df_resultados_horario.reset_index()

In [12]:
df_resultados_profesores = pd.DataFrame(data=config_tribunales, columns=['Presidente', 'Secretario', ''])
df_resultados = pd.concat([df_resultados_horario, df_resultados_profesores], axis=1)
df_resultados

,index,Fecha_Hora,Presidente,Secretario,
0,Tribunal_0,"[15, 15]",QYV,HLC,MSB
1,Tribunal_1,"[15, 15]",PMQ,EBB,IOE
2,Tribunal_2,"[15, 16]",RRD,QYV,MSB
3,Tribunal_3,"[15, 16]",PMQ,EBB,IOA
4,Tribunal_4,"[15, 17]",QYV,HLC,EBB
5,Tribunal_5,"[15, 17]",RRD,QWF,IOE
6,Tribunal_6,"[15, 18]",QYV,PMQ,IOA
7,Tribunal_7,"[15, 18]",LHL,MSB,QWF
8,Tribunal_8,"[15, 18]",RRD,EBB,IOE
9,Tribunal_9,"[15, 19]",HLC,EBB,IOA
